In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

transport = spark.table("scottish_housing_ws.silver.transport_accessibility")
postcodes = spark.table("scottish_housing_ws.silver.postcode_lookup")
hpi = spark.table("scottish_housing_ws.silver.uk_hpi")

In [0]:
transport.printSchema()
transport.show(3)

In [0]:
# Aggregate transport accessibility to council grain 
# One row per council area
# Metrics:
#   avg_bus_stops_500m: average number of bus stops within 500m per postcode
#   avg_rail_ferry_stops_1000m: average rail/ferry stops within 1000m
#   pct_zero_bus: percentage of postcodes with no bus stops within 500m
#   pct_zero_rail: percentage of postcodes with no rail/ferry within 1000m
#   pct_good_bus: percentage of postcodes with 5+ bus stops (well served)
#   pct_rail_connected: percentage of postcodes with at least 1 rail/ferry stop

transport_council = (
    transport
    .groupBy("council_area_code")
    .agg(
        F.count("postcode").alias("postcode_count"),
        F.round(F.avg("bus_stops_500m"), 2).alias("avg_bus_stops_500m"),
        F.round(F.avg("rail_ferry_stops_1000m"), 2).alias("avg_rail_ferry_stops_1000m"),
        F.round(
            F.sum(F.when(F.col("bus_stops_500m") == 0, 1).otherwise(0))
            / F.count("postcode") * 100, 1
        ).alias("pct_zero_bus"),
        F.round(
            F.sum(F.when(F.col("rail_ferry_stops_1000m") == 0, 1).otherwise(0))
            / F.count("postcode") * 100, 1
        ).alias("pct_zero_rail"),
        F.round(
            F.sum(F.when(F.col("bus_stops_500m") >= 5, 1).otherwise(0))
            / F.count("postcode") * 100, 1
        ).alias("pct_good_bus"),
        F.round(
            F.sum(F.when(F.col("rail_ferry_stops_1000m") >= 1, 1).otherwise(0))
            / F.count("postcode") * 100, 1
        ).alias("pct_rail_connected")
    )
)

In [0]:
latest_month = (
    hpi
    .filter(F.col("area_code").startswith("S12"))
    .groupBy("area_code")
    .agg(F.max("year_month").alias("latest_year_month"))
)

hpi_latest = (
    hpi
    .join(latest_month, on="area_code", how="inner")
    .filter(F.col("year_month") == F.col("latest_year_month"))
    .select(
        F.col("area_code").alias("council_area_code"),
        F.col("region_name").alias("council_name"),
        F.col("year_month").alias("hpi_year_month"),
        F.col("average_price").alias("council_avg_price")
    )
)

In [0]:
gold = (
    transport_council
    .join(hpi_latest, on="council_area_code", how="left")
    .select(
        "council_area_code",
        "council_name",
        "hpi_year_month",
        "council_avg_price",
        "postcode_count",
        "avg_bus_stops_500m",
        "avg_rail_ferry_stops_1000m",
        "pct_zero_bus",
        "pct_zero_rail",
        "pct_good_bus",
        "pct_rail_connected"
    )
    .orderBy(F.col("council_avg_price").desc())
)

In [0]:
print(f"Row count: {gold.count()}")
gold.show(32, truncate=False)

In [0]:
(
    gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("scottish_housing_ws.gold.transit_accessibility_vs_price")
)

In [0]:

(
    gold.coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("abfss://data@scottishhousingdl.dfs.core.windows.net/exports/gold_transit_accessibility_vs_price/")
)

print("Gold 12 complete.")

In [0]:
# Transit Accessibility vs Price
# Sources: silver.transport_accessibility, silver.postcode_lookup, silver.uk_hpi
# Grain: one row per council area, most recent HPI month

# Aggregates postcode-level transport accessibility scores to council area grain, then joins to council-level HPI.

# Limitation: no property-level transaction data available (RoS Cloudflare protected). Cannot compute per-postcode price premiums for transport accessibility. This table profiles council areas by transport access and correlates with council-level average prices.

# Transport metrics:
#   bus_stops_500m: active bus/coach stops within 500m of postcode centroid
#   rail_ferry_stops_1000m: active rail/ferry stops within 1000m
